In [16]:
# !pip install pandas nltk razdel spacy tabulate
# !python -m spacy download ru_core_news_sm

In [28]:
import pandas as pd
import re
import spacy
import nltk
from razdel import sentenize
from nltk.tokenize.punkt import PunktSentenceTokenizer
import random

# Preparation of tools
try:
    nlp_spacy = spacy.load("ru_core_news_sm")
except:
    import os
    os.system("python -m spacy download ru_core_news_sm")
    nlp_spacy = spacy.load("ru_core_news_sm")

nltk.download('punkt', quiet=True)
nltk_tokenizer = PunktSentenceTokenizer()

# --- 1. GENERATOR WITH DIFFERENT TYPES OF SEPARATORS ---

def generate_realistic_gold_standard(count=100):
    """Generates realistic medical records with proper boundary markings."""
    
    # Blocks indicating whether the end of the block is a sentence boundary
    blocks = [
        ("Жалобы: на боли в суставах.", True),
        (" Состояние: удовлетворительное", False),
        ("\nСознание: ясное", False),
        ("\nАД: 135/80 мм рт. ст.", False),
        (" ЧСС: 68 уд/мин, ритмичный.", True),
        (" Живот: мягкий, безболезненный; печень не увеличена.", True),
        (" Диагноз: ГБ II ст. риск 3.", True),
        ("\nРекомендовано: ФГДС; контроль АД", False),
        ("\nПринимает: конкор 5 мг; лозартан 100 мг", False),
        (" Объективно: кожа чистая, обычной окраски.", True),
        ("\nВ легких: дыхание везикулярное, хрипов нет", False),
        (" Терапия: коррекция дозы не требуется.", True),
        (" Пациент выписан под амбулаторное наблюдение.", True),
        ("\nРекомендации: диета с ограничением соли", False),
        (" Лабораторные данные: ОАК, ОАМ в норме.", True),
    ]
    
    gold_data = []
    for i in range(count):
        # Select 5-10 blocks randomly
        num_blocks = random.randint(5, 10)
        selected_blocks = random.sample(blocks, num_blocks)
        
        full_text = ""
        boundaries = []
        current_pos = 0
        
        for text_part, is_boundary in selected_blocks:
            full_text += text_part
            current_pos += len(text_part)
            if is_boundary:
                # If there is a boundary, add the end-of-sentence position
                boundaries.append(current_pos)
        
        # Remove the last border if it coincides with the end of the text
        if boundaries and boundaries[-1] >= len(full_text):
            boundaries.pop()
            
        gold_data.append({
            "text": full_text.strip(),
            "true_boundaries": [b for b in boundaries if b < len(full_text.strip())]
        })
    return gold_data

# --- 2. SEGMENTATION METHODS ---

def get_boundaries_baseline(text):
    """Basic Point Split (Medical Version)"""
    # Ignore periods in abbreviations - just split on periods with a space after
    boundaries = []
    
    # Look for periods followed by a space or the end of a line
    # and not preceded by a number or abbreviation
    for i, char in enumerate(text):
        if char == '.' and i < len(text) - 1:
            # Check if there is a space or the end of the text after the dot
            next_char = text[i + 1] if i + 1 < len(text) else ''
            if next_char in [' ', '\n', '\t', '']:
                # We check that there is no number or part of an abbreviation before the period
                if i > 0:
                    prev_chars = text[max(0, i-5):i]
                    # Skipping known abbreviations
                    is_abbreviation = any(abbr in prev_chars for abbr in 
                                        ['мм рт', 'уд/мин', 'мг', 'мл', 'см', 'г', 'ст'])
                    if not is_abbreviation:
                        # Check that there is no number before the period
                        if not text[i-1].isdigit():
                            boundaries.append(i + 1)  # position after dot
    
    if not boundaries:
        boundaries.append(len(text))
    elif boundaries[-1] < len(text):
        boundaries.append(len(text))
    
    return sorted(set(boundaries))

def get_boundaries_custom_med(text):
    """
    Custom medical segmenter.
    Accepts: \n, ; and periods after spaces.
    """
    boundaries = []
    
    # Adding borders for line breaks
    for match in re.finditer(r'\n+', text):
        boundaries.append(match.end())
    
    # Add borders by semicolons
    for match in re.finditer(r';', text):
        boundaries.append(match.start())
    
    # Adding boundaries by points (checking for abbreviations)
    # Finding all points
    for match in re.finditer(r'\.', text):
        pos = match.start()
        # Checking the context
        if pos > 0 and pos < len(text) - 1:
            # Skip abbreviations
            is_abbreviation = False
            
            # Check the previous characters for abbreviations
            context_start = max(0, pos - 10)
            context = text[context_start:pos]
            
            # List of abbreviations
            abbreviations = ['мм рт', 'уд/мин', 'мг', 'мл', 'см', 'г', 'ст', 'чсс', 'ад']
            for abbr in abbreviations:
                if abbr in context.lower():
                    is_abbreviation = True
                    break
            
            # Check if there is a space or end of line after the dot
            next_char = text[pos + 1] if pos + 1 < len(text) else ''
            if not is_abbreviation and next_char in [' ', '\n', '\t', '', '\r']:
                # Check that there is no number (not a decimal fraction) before the period
                if not text[pos-1].isdigit():
                    boundaries.append(pos + 1)  # include the dot
    
    boundaries.sort()
    if not boundaries or boundaries[-1] < len(text):
        boundaries.append(len(text))
    
    return sorted(list(set([b for b in boundaries if b <= len(text)])))

def get_boundaries_spacy(text):
    """spaCy with additional processing of medical abbreviations"""
    doc = nlp_spacy(text)
    boundaries = [sent.end_char for sent in doc.sents]
    
    # Correction for periods in abbreviations
    abbrev_pattern = r'\b(мм\.?рт\.?ст\.?|уд\.?/мин\.?|мг\.?|мл\.?|см\.?|г\.?)\b'
    for match in re.finditer(abbrev_pattern, text, re.IGNORECASE):
        # If the boundary coincides with the end of the contraction, we delete it
        end_pos = match.end()
        if end_pos in boundaries:
            boundaries.remove(end_pos)
    
    if not boundaries:
        boundaries = [len(text)]
    elif boundaries[-1] < len(text):
        boundaries.append(len(text))
    
    return boundaries

def get_boundaries_razdel(text):
    """Razdel with post-processing"""
    result = [sent.stop for sent in sentenize(text)]
    if not result:
        result = [len(text)]
    elif result[-1] < len(text):
        result.append(len(text))
    return result

def get_boundaries_nltk(text):
    """NLTK Punkt tokenizer"""
    spans = list(nltk_tokenizer.span_tokenize(text))
    if not spans:
        return [len(text)]
    boundaries = [span[1] for span in spans]
    if boundaries[-1] < len(text):
        boundaries.append(len(text))
    return boundaries

# --- 3. METRICS MEASUREMENT ---

def evaluate_metrics(true_b, pred_b, tolerance=2):
    """
    Segmentation quality assessment.
    tolerance: permissible deviation in characters (to account for small shifts)
    """
    if not true_b and not pred_b:
        return 0, 0, 0
    
    tp, fp, fn = 0, 0, 0
    matched_pred = set()
    
    for tb in true_b:
        # We are looking for the predicted boundary within tolerance
        match = next((pb for pb in pred_b if abs(tb - pb) <= tolerance 
                     and pb not in matched_pred), None)
        if match is not None:
            tp += 1
            matched_pred.add(match)
        else:
            fn += 1
    
    fp = len(pred_b) - len(matched_pred)
    return tp, fp, fn

# --- 4. VISUALIZATION OF RESULTS ---

def print_example_comparison(gold_item, tools_funcs):
    """Comparison output for a specific example"""
    print("\n" + "="*60)
    print("SEGMENTATION COMPARISON EXAMPLE")
    print("="*60)
    print(f"Text: {gold_item['text'][:200]}...")
    print(f"\nTrue boundaries (positions): {gold_item['true_boundaries']}")
    print("-"*60)
    
    for name, func in tools_funcs.items():
        pred_b = func(gold_item['text'])
        print(f"{name:20} -> {pred_b}")

# --- 5. LAUNCHING THE EXPERIMENT ---

# Generate test data
print("Generating synthetic medical records...")
gold_standard = generate_realistic_gold_standard(200)

tools = {
    "Baseline (Medical)": get_boundaries_baseline,
    "NLTK (Punkt)": get_boundaries_nltk,
    "spaCy (RU)": get_boundaries_spacy,
    "Razdel": get_boundaries_razdel,
    "Custom Medical": get_boundaries_custom_med
}

final_results = []

print("Evaluation of algorithms...")
for name, func in tools.items():
    t_tp, t_fp, t_fn = 0, 0, 0
    
    for item in gold_standard:
        try:
            pred_b = func(item["text"])
            tp, fp, fn = evaluate_metrics(item["true_boundaries"], pred_b, tolerance=2)
            t_tp += tp
            t_fp += fp
            t_fn += fn
        except Exception as e:
            print(f"Error in {name}: {e}")
            continue
        
    precision = t_tp / (t_tp + t_fp) if (t_tp + t_fp) > 0 else 0
    recall = t_tp / (t_tp + t_fn) if (t_tp + t_fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    final_results.append({
        "Algorithm": name,
        "F1-score": round(f1, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "TP (matched)": t_tp,
        "FP (extra)": t_fp,
        "FN (missed)": t_fn
    })

# Output results
df_res = pd.DataFrame(final_results).sort_values("F1-score", ascending=False)
print("\n" + "="*80)
print("COMPARISON OF SEGMENTATION TOOLS FOR MEDICAL DATA")
print("="*80)
print(df_res.to_string(index=False))

# We show an example for verification
print("\n" + "="*80)
print("EXAMPLE OF A GENERATED RECORD")
print("="*80)
example = gold_standard[0]
print(f"Text:\n{example['text']}\n")
print(f"True sentence boundaries (symbol positions): {example['true_boundaries']}")

# Demonstration of partitioning for the first example with all tools
print("\n" + "="*80)
print("DEMONSTRATION OF SENTENCE BREAKING")
print("="*80)
for name, func in tools.items():
    try:
        pred_b = func(example['text'])
        print(f"\n{name}:")
        print(f"Boundaries: {pred_b}")
        # Show splitted text
        start = 0
        for i, bound in enumerate(pred_b):
            if bound <= len(example['text']):
                sentence = example['text'][start:bound].strip()
                if sentence:  # Show only non-empty sentences
                    print(f"  {i+1}. {sentence}")
                start = bound
    except Exception as e:
        print(f"\n{name}: Error - {e}")

Generating synthetic medical records...
Evaluation of algorithms...

COMPARISON OF SEGMENTATION TOOLS FOR MEDICAL DATA
         Algorithm  F1-score  Precision  Recall  TP (matched)  FP (extra)  FN (missed)
            Razdel    0.8350     0.7605  0.9257           635         200           51
        spaCy (RU)    0.7955     0.6690  0.9810           673         333           13
Baseline (Medical)    0.7694     0.7347  0.8076           554         200          132
      NLTK (Punkt)    0.7099     0.5757  0.9257           635         468           51
    Custom Medical    0.5077     0.3741  0.7901           542         907          144

EXAMPLE OF A GENERATED RECORD
Text:
Живот: мягкий, безболезненный; печень не увеличена. Диагноз: ГБ II ст. риск 3.
Рекомендации: диета с ограничением соли Терапия: коррекция дозы не требуется. Объективно: кожа чистая, обычной окраски.
Рекомендовано: ФГДС; контроль АД ЧСС: 68 уд/мин, ритмичный.
В легких: дыхание везикулярное, хрипов нет Состояние: удовлетво